In [1]:
import pandas as pd

business = pd.read_csv(r"C:\Users\Rudra\data analytics\Ecommerce_Revenue_Intelligence\data\business_master.csv")

In [2]:
print('Shape:', business.shape)

business.isnull().sum().sort_values(ascending=False)

Shape: (99441, 15)


order_delivered_customer_date    2965
order_delivered_carrier_date     1783
revenue                           775
freight                           775
total_revenue                     775
order_approved_at                 160
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
dtype: int64

In [3]:
business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99441 non-null  object 
 1   customer_id                    99441 non-null  object 
 2   order_status                   99441 non-null  object 
 3   order_purchase_timestamp       99441 non-null  object 
 4   order_approved_at              99281 non-null  object 
 5   order_delivered_carrier_date   97658 non-null  object 
 6   order_delivered_customer_date  96476 non-null  object 
 7   order_estimated_delivery_date  99441 non-null  object 
 8   customer_unique_id             99441 non-null  object 
 9   customer_zip_code_prefix       99441 non-null  int64  
 10  customer_city                  99441 non-null  object 
 11  customer_state                 99441 non-null  object 
 12  revenue                        98666 non-null 

In [4]:
# Check duplicate order_id
print('Duplicate orders:', business['order_id'].duplicated().sum())

Duplicate orders: 0


In [5]:
# Check missing revenue
business[['revenue', 'freight', 'total_revenue']].isnull().sum()

revenue          775
freight          775
total_revenue    775
dtype: int64

In [6]:
business['revenue'] = business['revenue'].fillna(0)
business['freight'] = business['freight'].fillna(0)
business['total_revenue'] = business['total_revenue'].fillna(0)

In [7]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    business[col] = pd.to_datetime(business[col], errors='coerce')

business[date_cols].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [8]:
business['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
print('Negative revenue:', (business['total_revenue'] < 0).sum())
print('Zero revenue:', (business['total_revenue'] == 0).sum())

Negative revenue: 0
Zero revenue: 775


In [10]:
business = business[business['order_status'] == 'delivered'].copy()
print(business.shape)

(96478, 15)


In [11]:
text_cols = ['customer_city', 'customer_state']

for col in text_cols:
    business[col] = business[col].astype(str).str.strip().str.title()

business[text_cols].head()

,customer_city,customer_state
0,Sao Paulo,Sp
1,Barreiras,Ba
2,Vianopolis,Go
3,Sao Goncalo Do Amarante,Rn
4,Santo Andre,Sp


In [12]:
business['delivery_days'] = (
    business['order_delivered_customer_date'] -
    business['order_purchase_timestamp']
).dt.days

business['estimated_days'] = (
    business['order_estimated_delivery_date'] -
    business['order_purchase_timestamp']
).dt.days

business['delay_days'] = (
    business['order_delivered_customer_date'] -
    business['order_estimated_delivery_date']
).dt.days

In [13]:
print('Rows:', business.shape[0])
print('Columns:', business.shape[1])

print('\nMissing values:')
print(business.isnull().sum()[business.isnull().sum() > 0])

print('\nRevenue summary:')
print(business['total_revenue'].describe())

Rows: 96478
Columns: 18

Missing values:
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
delivery_days                     8
delay_days                        8
dtype: int64

Revenue summary:
count    96478.000000
mean       159.826839
std        218.794219
min          9.590000
25%         61.850000
50%        105.280000
75%        176.260000
max      13664.080000
Name: total_revenue, dtype: float64


In [14]:
print('Unique customers:', business['customer_unique_id'].nunique())
print('Unique orders:', business['order_id'].nunique())
print('Total revenue:', round(business['total_revenue'].sum(), 2))

Unique customers: 93358
Unique orders: 96478
Total revenue: 15419773.75


In [15]:
business.to_csv('../data/business_master_cleaned.csv', index=False)

print('Cleaned business master table saved!')

Cleaned business master table saved!
